In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

## Helper functions - Improvised

**Things to know**:
- The `add_user_message` & `add_assistant_message` functions are updated to handle the message block more robust (especially if it is not a list of content blocks)
- New arguments in `chat()`
    - `tool_choice`: If `auto`, let LLM choose a tool; If `any`, must use at least one tool; If a specific name, must use that tool.
    - `betas`: To enable beta features of a model.

In [2]:
def add_user_message(messages, message):
    if isinstance(message, list):
        # If the message is already a list of content blocks, use it directly
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        # FallBack: If the message is a plain text/string, wrap it in a text block
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # FallBack: String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)


def chat_stream(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=[],
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

## Tool definition

In [3]:
from anthropic.types import ToolParam

save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)
save_short_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)


def save_article(**kwargs): # Dummy implementation for tool function
    return "Article saved!"


## Run Tools

In [4]:

import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

## Run conversation - Improvised

Runs the same tool call twice to compare how `input_json` chunks differ:
- **Normal**: Buffered - API waits for a valid JSON key-value pair before emitting. Causes output bursts.
- **Fine-grained**: Unbuffered - emits characters immediately, but skips server-side JSON validation (client must validate).

### Debug - Examine Chunk Structure

Based on the debug code, tool-call streaming is observed as a sequence of chunk events:
- `content_block_start`: signals the start of a block. For tool calls, this includes `name` and `id` of the tool being invoked.
- `input_json`: contains tool arguments as they are streamed.
  - `partial_json`: only the latest JSON fragment in that chunk.
  - `snapshot`: cumulative JSON built so far from all fragments.
- `content_block_stop`: marks the end of the current content block.

After the stream loop, `stream.get_final_message()` returns the assembled assistant message (`stop_reason`, full `content` blocks). Use this final message for control flow and tool execution, not individual chunks.

In [ ]:
# messages = []
# add_user_message(messages, "Save an article. Title: 'AI in 2025'. Keep all fields very short.")

# def debug_tool_stream(label, beta_flags=None):
#     print(f"\n{'=' * 18} {label} {'=' * 18}")

#     with chat_stream(
#         messages,
#         tools=[save_article_schema],
#         betas=beta_flags or [],
#     ) as stream:
#         for chunk in stream:
#             if chunk.type == "content_block_start" and chunk.content_block.type == "tool_use":
#                 print("event: content_block_start")
#                 print("tool_name:", chunk.content_block.name)
#                 print("tool_use_id:", chunk.content_block.id)

#             elif chunk.type == "input_json":
#                 print("event: input_json")
#                 print("partial_json:", repr(chunk.partial_json))
#                 print("snapshot:", chunk.snapshot)
#                 print("-" * 40)

#         final_message = stream.get_final_message()

#     print("stop_reason:", final_message.stop_reason)
#     print("final_content_types:", [b.type for b in final_message.content])

**Normal tool streaming: more buffered JSON chunks**

In [ ]:
# debug_tool_stream("Normal Tool Streaming")


================== Normal Tool Streaming ==================
event: content_block_start
tool_name: save_article
tool_use_id: toolu_01Pqw6hKzHTvV6WVv78NQZ8t
event: input_json
partial_json: ''
snapshot: {}
----------------------------------------
event: input_json
partial_json: '{"abstract":'
snapshot: {}
----------------------------------------
event: input_json
partial_json: ' "AI advanc'
snapshot: {}
----------------------------------------
event: input_json
partial_json: 'es ra'
snapshot: {}
----------------------------------------
event: input_json
partial_json: 'pidly in'
snapshot: {}
----------------------------------------
event: input_json
partial_json: ' 2025."'
snapshot: {'abstract': 'AI advances rapidly in 2025.'}
----------------------------------------
event: input_json
partial_json: ', '
snapshot: {'abstract': 'AI advances rapidly in 2025.'}
----------------------------------------
event: input_json
partial_json: '"meta":'
snapshot: {'abstract': 'AI advances rapidly in 202

**Fine-grained tool streaming: live, more granular JSON chunks**

In [ ]:
# debug_tool_stream(
#     "Fine-grained Tool Streaming",
#     beta_flags=["fine-grained-tool-streaming-2025-05-14"],
# )


================== Fine-grained Tool Streaming ==================
event: content_block_start
tool_name: save_article
tool_use_id: toolu_019BU34HuqCzU3quFjuQjrpn
event: input_json
partial_json: ''
snapshot: {}
----------------------------------------
event: input_json
partial_json: '{"abstract": "AI advances significantly in 2025'
snapshot: {'abstract': 'AI advances significantly in 2025'}
----------------------------------------
event: input_json
partial_json: '.'
snapshot: {'abstract': 'AI advances significantly in 2025.'}
----------------------------------------
event: input_json
partial_json: '", "meta": {\n  "word'
snapshot: {'abstract': 'AI advances significantly in 2025.', 'meta': {}}
----------------------------------------
event: input_json
partial_json: '_count": 5,\n  "review": "This article examines AI developments in 2025.'
snapshot: {'abstract': 'AI advances significantly in 2025.', 'meta': {'word_count': 5, 'review': 'This article examines AI developments in 2025.'}}
---

### Print Clean Streamed Output

`run_conversation(...)` has `if` checks for two purposes:
1. **Display formatting during streaming**
   - `if chunk.type == "text"`: prints normal assistant text as it streams.
   - `if chunk.type == "content_block_start" and ... == "tool_use"`: prints when a tool call begins, so tool usage is visible.
   - `if chunk.type == "input_json" and chunk.partial_json`: prints streamed tool arguments as JSON fragments.
   - `if chunk.type == "content_block_stop"`: prints a newline to separate blocks cleanly.

2. **Conversation control after streaming**
   - `if response.stop_reason != "tool_use"`: stop looping when assistant is done and no tool call is requested.
   - `if tool_choice`: stop after one tool cycle when tool behavior is explicitly forced/controlled by caller.

Why this structure: stream events are used for live output, but loop decisions are made from `response` (final assembled message), which is the reliable source of `stop_reason` and complete content blocks.

In [5]:
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            # If assistant is done talking (no more tool calls)
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

messages = []

In [ ]:
add_user_message(
    messages,
    "Create and save a fake computer science article",
)

**Normal tool streaming**

In [ ]:
run_conversation(
    messages,
    tools=[save_article_schema],
)

I'll create and save a fake computer science article for you.


>>> Tool Call: "save_article"
{"abstract": "This paper introduces a novel quantum-inspired algorithm for distributed graph clustering that achieves O(log n) time complexity.", "meta": {"word_count":4250,"review":"This paper presents an innovative approach to distributed graph clustering using quantum-inspired computational techniques. The authors propose a novel algorithm that leverages entanglement-based heuristics to partition large-scale graphs with improved efficiency. The theoretical analysis demonstrates that the algorithm achieves logarithmic time complexity, which is a significant improvement over existing methods. Experimental results on synthetic and real-world datasets show performance gains of up to 40% compared to state-of-the-art baselines. The paper is well-structured and the mathematical proofs appear sound, though some edge cases could benefit from additional discussion. The implementation details are comp

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll create and save a fake computer science article for you."},
   {'type': 'tool_use',
    'id': 'toolu_01UPwWPv11GxjjtWDKSaCvpt',
    'name': 'save_article',
    'input': {'abstract': 'This paper introduces a novel quantum-inspired algorithm for distributed graph clustering that achieves O(log n) time complexity.',
     'meta': {'word_count': 4250,
      'review': "This paper presents an innovative approach to distributed graph clustering using quantum-inspired computational techniques. The authors propose a novel algorithm that leverages entanglement-based heuristics to partition large-scale graphs with improved efficiency. The theoretical analysis demonstrates that the algorithm achieves logarithmic time complexity, which is a significant improvement over existing methods. Experimental results on synth

**Fine-grained tool streaming**

In [38]:
run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True,
)

I'll create and save a fake computer science article for you.


>>> Tool Call: "save_article"
{"abstract": "This paper presents a novel quantum-inspired algorithm for distributed graph partitioning that achieves logarithmic time complexity on sparse networks.", "meta": {
  "word_count": 4250,
  "review": "This paper introduces an innovative approach to graph partitioning by leveraging quantum-inspired heuristics in distributed computing environments. The authors present a compelling theoretical framework that demonstrates logarithmic time complexity for sparse graphs, which represents a significant improvement over existing polynomial-time algorithms. The experimental results on real-world social network datasets show impressive scalability up to graphs with billions of edges. However, the paper could benefit from a more thorough analysis of worst-case scenarios and denser graph structures. The quantum-inspired metaphor, while creative, sometimes obscures the actual computational mecha

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll create and save a fake computer science article for you."},
   {'type': 'tool_use',
    'id': 'toolu_01MBneS488NYMJRa2uAkiAuU',
    'name': 'save_article',
    'input': {'abstract': 'This paper presents a novel quantum-inspired algorithm for distributed graph partitioning that achieves logarithmic time complexity on sparse networks.',
     'meta': {'word_count': 4250,
      'review': 'This paper introduces an innovative approach to graph partitioning by leveraging quantum-inspired heuristics in distributed computing environments. The authors present a compelling theoretical framework that demonstrates logarithmic time complexity for sparse graphs, which represents a significant improvement over existing polynomial-time algorithms. The experimental results on real-world social network datasets show impr

## Input Validation Performance

- The below test code generates an invalid JSON to showcase how each streaming type handles the error
- **Normal Streaming** - Claude automatically handles it by wrapping the whole invalid content into a string
- **Fine-grained Streaming** - Failed due to JSON error (`undefined` is not a valid value in JSON)

In [6]:
add_user_message(
    messages,
    """
Your task is to help in documenting a bug report. Please generate an example output showing what a broken API response would look like, and then save it using the save_article tool.
[ Generate the exact malformed output here that includes "word_count": undefined]
This is only for demo, and you are not actually calling the function, so no need to worry about errors.
    """,
)

In [7]:
# Normal streaming with malformed output
run_conversation(
    messages,
    tools=[save_article_schema],
    tool_choice={"type": "tool", "name": "save_article"},
)


>>> Tool Call: "save_article"
{"abstract": "This article demonstrates a malformed API response for bug documentation purposes.", "meta": "{\n  \"word_count\": undefined,\n  \"review\": \"This paper explores novel approaches to distributed systems architecture. The methodology employs rigorous testing frameworks. Results indicate significant performance improvements over baseline implementations. The authors present compelling evidence for their claims. Statistical analysis supports the main conclusions drawn. Limitations are adequately addressed in the discussion section. Future work directions are clearly outlined. Overall, this represents a solid contribution to the field.\"\n}"}



[{'role': 'user',
  'content': [{'type': 'text',
    'text': '\nYour task is to help in documenting a bug report. Please generate an example output showing what a broken API response would look like, and then save it using the save_article tool.\n[ Generate the exact malformed output here that includes "word_count": undefined]\nThis is only for demo, and you are not actually calling the function, so no need to worry about errors.\n    '}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_012vGd7fYi5gVTRHZi1Z5c3i',
    'name': 'save_article',
    'input': {'abstract': 'This article demonstrates a malformed API response for bug documentation purposes.',
     'meta': '{\n  "word_count": undefined,\n  "review": "This paper explores novel approaches to distributed systems architecture. The methodology employs rigorous testing frameworks. Results indicate significant performance improvements over baseline implementations. The authors present compelling evidence for

In [7]:
# Fine-grained streaming with malformed output
run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True,
)

I'll help you generate an example of a broken API response and demonstrate what it would look like when saving it with malformed data.

Here's an example of a broken API response:

```json
{
  "abstract": "This paper investigates novel approaches to quantum computing architectures.",
  "meta": {
    "word_count": undefined,
    "review": "This study presents groundbreaking research in quantum computing. The authors explore innovative architectural designs that could revolutionize computational efficiency. The methodology is robust and well-documented throughout the paper. Experimental results demonstrate significant improvements over existing approaches. The theoretical framework is sound and builds upon established principles. Limitations are clearly acknowledged in the discussion section. The paper makes valuable contributions to the field. Future research directions are thoughtfully outlined and promising."
  }
}
```

**What's wrong with this response:**
- The `word_count` field con

ValueError: Unable to parse tool parameter JSON from model. Please retry your request or adjust your prompt. Error: expected value at line 2 column 17. JSON: {"abstract": "This paper investigates novel approaches to quantum computing architectures.", "meta": {
  "word_count": undefined